# TransferZAI — Cross-Encoder Fine-Tuning (v2)

Fine-tunes a cross-encoder reranker on transfer equivalency pairs from W&M, VT, and CCC→UCSC.

**Key improvements over v1:**
- **Catalog augmentation**: candidate text includes known CC equivalent titles as anchor text
  (e.g. THEA 30 description + 8 CCC theater course titles that map to it). Only applied when
  3+ CC equivalents exist, exploiting the many-to-one structure of the data.
- **Full-catalog negatives**: negatives sampled from the entire university catalog, not same-dept.
  Same-dept is wrong here because VCCS/CCC dept codes never align with university dept codes.
- **Cross-institutional transitivity**: VCCS courses appearing in both W&M and VT tables generate
  extra training signal (the shared VCCS course links the two university courses).
- **Clean train/test split**: 20% held-out test rows excluded from training.

**Runtime:** GPU (T4 is fine). Runtime → Change runtime type → T4 GPU.

**Output:** Downloads `cross_encoder.zip` at the end. Unzip into your project root.

In [ ]:
# ── Install dependencies ─────────────────────────────────────
!pip install -q sentence-transformers pandas scikit-learn

In [ ]:
import os

# ── Verify files are present (dragged into the files tab) ────
required = [
    'vccs_wm_merged.csv',
    'vccs_vt_merged.csv',
    'ccc_ucsc_clean.csv',
    'wm_courses_2025.csv',
    'vt_courses_2025.csv',
    'ucsc_courses_2025.csv',
]
for f in required:
    status = '✓' if os.path.exists(f) else '✗ MISSING'
    print(f'  {status}  {f}')

In [ ]:
import re, random, os
import numpy as np
import pandas as pd
import torch
from collections import defaultdict
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sentence_transformers import InputExample
from sentence_transformers.cross_encoder import CrossEncoder

random.seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

NEGS_PER_POS = 6   # increased from 4 — full-catalog negatives are easier, need more variety

In [ ]:
# ── Helpers ──────────────────────────────────────────────────
_CCC_NOISE = re.compile(
    r'may be offered in a distance.learning format'
    r'|transfer information:|transferable to (?:both )?uc'
    r'|limitations on enrollment:|requisites:'
    r'|minimum credit units|maximum credit units'
    r'|toggle (?:additional|general|learning)'
    r'|grade options:|see open sections|connect with an academic counselor'
    r'|some of the class hours for this course',
    re.IGNORECASE
)
_LAB_CODE  = re.compile(r'[A-Z]+\s+\d+L$')
_LAB_TEXT  = re.compile(r'\b(lab|laboratory)\b', re.I)
_SEQ_CODE  = re.compile(r'([A-Z]+)\s+\d+([A-D])L?$')
_SEQ_TITLE = [
    (1, re.compile(r'\b[1I]\b\s*$')),
    (2, re.compile(r'\b(?:2|II)\b\s*$')),
    (3, re.compile(r'\b(?:3|III)\b\s*$')),
    (4, re.compile(r'\b(?:4|IV)\b\s*$')),
]
_SEQ_DESC  = [
    (2, re.compile(r'\bcontinuation\s+of\b', re.I)),
    (1, re.compile(r'\bfirst\s+(?:course|quarter|semester|part)\b', re.I)),
    (2, re.compile(r'\bsecond\s+(?:course|quarter|semester|part)\b', re.I)),
    (3, re.compile(r'\bthird\s+(?:course|quarter|semester|part)\b', re.I)),
    (3, re.compile(r'\bfinal\s+(?:course|quarter|semester|part)\b', re.I)),
]
_SEQ_LABELS = {1: 'first in sequence', 2: 'second in sequence',
               3: 'third in sequence', 4: 'fourth in sequence'}

def clean_text(text):
    if pd.isna(text) or str(text) in ('Description not found', 'nan', ''): return ''
    text = str(text)
    m = _CCC_NOISE.search(text)
    if m: text = text[:m.start()]
    text = text.lower()
    text = re.sub(r'prerequisite\(s\):.*?(?=\.\s|$)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'cross-listed with:.*?(?=\.\s|$)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)  # keep digits — they carry level/sequence info
    return re.sub(r'\s+', ' ', text).strip()

def seq_token_from_code(code):
    m = _SEQ_CODE.match(str(code).strip())
    if not m: return None
    return _SEQ_LABELS.get({'A':1,'B':2,'C':3,'D':4}.get(m.group(2)))

def seq_token_from_text(title, desc=''):
    for pos, pat in _SEQ_TITLE:
        if pat.search(title.strip()): return _SEQ_LABELS[pos]
    for pos, pat in _SEQ_DESC:
        if pat.search(str(desc)): return _SEQ_LABELS[pos]
    return None

def lab_token_candidate(code, title, desc):
    if _LAB_CODE.match(str(code).strip()) and (_LAB_TEXT.search(title) or _LAB_TEXT.search(desc)):
        return 'laboratory course'
    return None

def lab_token_query(title):
    return 'laboratory course' if _LAB_TEXT.search(title) else None

def augment(text, *tokens):
    for t in tokens:
        if t: text = f'{text} {t}'
    return text.strip()

def load_catalog(filename, encoding='utf-8'):
    df = pd.read_csv(filename, encoding=encoding).dropna(subset=['course_code'])
    lk = {}
    for _, r in df.iterrows():
        code = str(r['course_code']).strip()
        lk[code] = {
            'title': str(r.get('course_title', '')),
            'description': str(r.get('course_description', '')) if pd.notna(r.get('course_description')) else '',
        }
    return lk

def candidate_text_clean(code, title, desc):
    """Clean text — baseline representation."""
    base = f'{title} {clean_text(desc)}'
    return augment(base, seq_token_from_code(code), lab_token_candidate(code, title, desc))

def candidate_text_aug(code, title, desc, equiv):
    """Augmented text — appends known CC equivalent titles when 3+ exist.
    This aligns the training distribution with the eval pipeline,
    where catalog augmentation significantly improves BGE retrieval."""
    base = candidate_text_clean(code, title, desc)
    if code in equiv and len(equiv[code]) >= 3:
        anchors = ' '.join(equiv[code][:8])
        base = f'{base} {anchors}'
    return base.strip()

def sample_negatives(target_code, lookup, n):
    """Sample n negatives from the full catalog (excluding target).
    Note: same-dept sampling is wrong here — VCCS/CCC dept codes never
    align with university dept codes (empirically <1% overlap)."""
    all_codes = [c for c in lookup if c != target_code]
    return random.sample(all_codes, min(n, len(all_codes)))

def parse_vccs(raw):
    parts = re.split(r'\s*TAKEN WITH\s*', str(raw).strip(), flags=re.IGNORECASE)
    courses = []
    for p in parts:
        m = re.match(r'([A-Z]{2,4})\s+(\d{3})\s+(.*)', p.strip())
        if m:
            courses.append({'dept': m.group(1), 'title': m.group(3).strip()})
    return courses

def vccs_query_text(title, desc):
    base = f'{title} {clean_text(desc)}'
    return augment(base, seq_token_from_text(title, desc))

def ccc_query_text(title, desc):
    base = f'{title} {clean_text(desc)}'
    return augment(base, seq_token_from_text(title, desc), lab_token_query(title))

print('Helpers defined.')

In [ ]:
# ── Load catalogs ────────────────────────────────────────────
wm_lookup   = load_catalog('wm_courses_2025.csv', encoding='latin-1')
vt_lookup   = load_catalog('vt_courses_2025.csv')
ucsc_lookup = load_catalog('ucsc_courses_2025.csv')
print(f'W&M: {len(wm_lookup)} | VT: {len(vt_lookup)} | UCSC: {len(ucsc_lookup)}')

In [ ]:
# ── Build training pairs (train split only — no leakage) ─────
# Each dataset is split 80/20 with the same random_state=42 used in eval,
# so the held-out 20% test rows are never seen during CE training.

def wm_train_split(df):
    has = df['W&M Course Code'].notna() & (df['W&M Course Code'].str.strip() != '')
    pos = df[has].copy()
    def _dept(c):
        m = re.match(r'([A-Z]{2,4})\s+\d+', str(c).strip())
        return m.group(1) if m else 'UNK'
    pos['_dept'] = pos['W&M Course Code'].apply(_dept)
    dc = pos['_dept'].value_counts()
    pos['_strat'] = pos['_dept'].apply(lambda d: 'RARE' if dc[d] < 2 else d)
    train, _ = train_test_split(pos, test_size=0.20, random_state=42, stratify=pos['_strat'])
    return train

def vt_train_split(df):
    def _dept(c):
        m = re.match(r'([A-Z]+)\s+\d+', str(c).strip())
        return m.group(1) if m else 'UNK'
    df = df.copy()
    df['_dept'] = df['VT Course Code'].apply(lambda x: _dept(x.split('+')[0].strip()))
    dc = df['_dept'].value_counts()
    df['_strat'] = df['_dept'].apply(lambda d: 'RARE' if dc[d] < 2 else d)
    train, _ = train_test_split(df, test_size=0.20, random_state=42, stratify=df['_strat'])
    return train

def ccc_train_split(df):
    def _dept(c):
        m = re.match(r'([A-Z]+)\s+', str(c).strip())
        return m.group(1) if m else 'UNK'
    df = df.copy()
    df['_dept'] = df['UCSC Course Code'].apply(_dept)
    dc = df['_dept'].value_counts()
    df['_strat'] = df['_dept'].apply(lambda d: 'RARE' if dc[d] < 2 else d)
    train, _ = train_test_split(df, test_size=0.20, random_state=42, stratify=df['_strat'])
    return train

# Load datasets and split
df_wm = pd.read_csv('vccs_wm_merged.csv')
df_wm.columns = df_wm.columns.str.strip()
wm_train = wm_train_split(df_wm)
print(f'W&M  train rows: {len(wm_train)}')

df_vt = pd.read_csv('vccs_vt_merged.csv')
vt_train = vt_train_split(df_vt)
print(f'VT   train rows: {len(vt_train)}')

df_ccc = pd.read_csv('ccc_ucsc_clean.csv')
df_ccc.columns = df_ccc.columns.str.strip()
ccc_train = ccc_train_split(df_ccc)
print(f'CCC  train rows: {len(ccc_train)}')

# ── Build equiv maps from training data only ──────────────────
# {university_code: [list of CC course titles from training equivalencies]}
# Used to augment candidate text, pulling university embeddings toward CC query space.

wm_equiv = defaultdict(list)
for _, row in wm_train.iterrows():
    target = str(row['W&M Course Code']).strip()
    for c in parse_vccs(row['VCCS Course']):
        wm_equiv[target].append(c['title'])
wm_equiv = dict(wm_equiv)

vt_equiv = defaultdict(list)
for _, row in vt_train.iterrows():
    target = row['VT Course Code'].split('+')[0].strip()
    for c in parse_vccs(row['VCCS Course']):
        vt_equiv[target].append(c['title'])
vt_equiv = dict(vt_equiv)

ccc_equiv = defaultdict(list)
for _, row in ccc_train.iterrows():
    target = str(row['UCSC Course Code']).strip()
    title_raw = str(row.get('CCC Course', '')).strip()
    m = re.match(r'^[A-Z][A-Z0-9 ]*?\s+\S+\s+(.*)', title_raw)
    title = m.group(1).strip() if m else title_raw
    if title:
        ccc_equiv[target].append(title)
ccc_equiv = dict(ccc_equiv)

wm_aug_count  = sum(1 for v in wm_equiv.values()  if len(v) >= 3)
vt_aug_count  = sum(1 for v in vt_equiv.values()  if len(v) >= 3)
ccc_aug_count = sum(1 for v in ccc_equiv.values() if len(v) >= 3)
print(f'\nCourses augmented (3+ CC equivalents):')
print(f'  W&M: {wm_aug_count}/{len(wm_equiv)}')
print(f'  VT:  {vt_aug_count}/{len(vt_equiv)}')
print(f'  CCC: {ccc_aug_count}/{len(ccc_equiv)}')

# ── Build training examples ───────────────────────────────────
examples = []

# W&M
for _, row in wm_train.iterrows():
    target = str(row['W&M Course Code']).strip()
    if target not in wm_lookup: continue
    courses = parse_vccs(row['VCCS Course'])
    titles  = ' '.join(c['title'] for c in courses)
    query   = vccs_query_text(titles, row.get('VCCS Description', ''))
    pos     = candidate_text_aug(target, wm_lookup[target]['title'],
                                  wm_lookup[target]['description'], wm_equiv)
    examples.append(InputExample(texts=[query, pos], label=1.0))
    for neg in sample_negatives(target, wm_lookup, NEGS_PER_POS):
        examples.append(InputExample(
            texts=[query, candidate_text_aug(neg, wm_lookup[neg]['title'],
                                             wm_lookup[neg]['description'], wm_equiv)],
            label=0.0))

# VT
for _, row in vt_train.iterrows():
    target = row['VT Course Code'].split('+')[0].strip()
    if target not in vt_lookup: continue
    courses = parse_vccs(row['VCCS Course'])
    titles  = ' '.join(c['title'] for c in courses)
    query   = vccs_query_text(titles, row.get('VCCS Description', ''))
    vt_title = str(row.get('VT Course Title', ''))
    vt_desc  = str(row.get('VT Description', '')).split('|')[0]
    pos      = candidate_text_aug(target, vt_title, vt_desc, vt_equiv)
    examples.append(InputExample(texts=[query, pos], label=1.0))
    for neg in sample_negatives(target, vt_lookup, NEGS_PER_POS):
        examples.append(InputExample(
            texts=[query, candidate_text_aug(neg, vt_lookup[neg]['title'],
                                             vt_lookup[neg]['description'], vt_equiv)],
            label=0.0))

# CCC → UCSC
for _, row in ccc_train.iterrows():
    target = str(row['UCSC Course Code']).strip()
    if target not in ucsc_lookup: continue
    title_raw = str(row.get('CCC Course', '')).strip()
    m = re.match(r'^[A-Z][A-Z0-9 ]*?\s+\S+\s+(.*)', title_raw)
    title = m.group(1).strip() if m else title_raw
    query = ccc_query_text(title, row.get('CCC Description', ''))
    pos   = candidate_text_aug(target, str(row.get('UCSC Course Title', '')),
                               str(row.get('UCSC Description', '')), ccc_equiv)
    examples.append(InputExample(texts=[query, pos], label=1.0))
    for neg in sample_negatives(target, ucsc_lookup, NEGS_PER_POS):
        examples.append(InputExample(
            texts=[query, candidate_text_aug(neg, ucsc_lookup[neg]['title'],
                                             ucsc_lookup[neg]['description'], ccc_equiv)],
            label=0.0))

random.shuffle(examples)
n_pos = sum(1 for e in examples if e.label == 1.0)
n_neg = sum(1 for e in examples if e.label == 0.0)
print(f'\nTraining examples: {len(examples)}  (pos={n_pos}, neg={n_neg}, ratio={n_neg/n_pos:.1f}:1)')
print(f'(Excluded ~20% test rows from each dataset to prevent leakage)')

In [ ]:
# ── Fine-tune ────────────────────────────────────────────────
ce_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', num_labels=1, device=device)

loader = DataLoader(examples, shuffle=True, batch_size=32)
EPOCHS = 12
warmup = int(len(loader) * EPOCHS * 0.1)
print(f'Fine-tuning: {EPOCHS} epochs, {len(loader)} steps/epoch, warmup={warmup}')

ce_model.fit(
    train_dataloader=loader,
    epochs=EPOCHS,
    warmup_steps=warmup,
    show_progress_bar=True,
    use_amp=True,   # FP16 on GPU for speed
)

os.makedirs('cross_encoder', exist_ok=True)
ce_model.save('cross_encoder')
print('Saved to ./cross_encoder/')

In [ ]:
from google.colab import files
!zip -r cross_encoder.zip cross_encoder/
files.download('cross_encoder.zip')
print('Done. Unzip into your transferzaidemo/ project root.')